### Compute power, sensitivity and PPV for results obtained over groups of datasets

In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from tqdm import tqdm

import folium
import branca.colormap as cm

In [ ]:
# Read the flattened candidates.
path_dict_candidates = './data_simulator/huge_dataset/gencand/dict_flattened_candidates.pkl'
with open(path_dict_candidates, "rb") as f:
    dict_candidates = pickle.load(f)
# display(dict_candidates)

# Read the geodataframes of the grids (needed to plot the results on a map).
path_dict_grids = './data_simulator/huge_dataset/grids/dict_grids.pkl'
with open(path_dict_grids, "rb") as f:
    dict_grids = pickle.load(f)

For each candidate, retrieve the grid and subset of cell it refers to.

In [ ]:
# Retrieve the set of candidates (subset of cells) that underwent hypothesis testing.
grid_info = dict_candidates['grid_info']
# display(grid_info)

# Retrieve the flattened list of object IDs associated with the candidates.
flattened_list_candidates = dict_candidates['flat_ids']

# Compute the starting pos, ending pos, and number of objects of each candidate.
start_pos_candidates = dict_candidates['start_pos'][:-1]
end_pos_candidates = dict_candidates['start_pos'][1:]
num_objs_candidates = np.diff(dict_candidates['start_pos'])

# Some DEBUG.
# print(len(start_pos_candidates), len(end_pos_candidates), len(num_objs_candidates))
# print(start_pos_candidates[-4], end_pos_candidates[-5])

In [ ]:
# For each candidate, find out the resolution of the grid it comes from.
num_candidates = start_pos_candidates.size
candidates_grid_res = np.empty(num_candidates, dtype=np.uint32)
count = 0
for grid in grid_info:
    cell_ids = grid[3].to_numpy()
    num_els_grid = cell_ids.size

    grid_id = np.empty(1, dtype=np.uint32)
    grid_id[0] = grid[1]
    grid_id = np.repeat(grid_id, cell_ids.size)    
    
    candidates_grid_res[count : count + num_els_grid] = grid_id

    count += num_els_grid

# Create the list of grid resultions used during the assessment.
# NOTE: the zero resolution is used to consider the candidates from ALL the grids.
list_grid_res = [0] + np.unique(candidates_grid_res).tolist()


display(list_grid_res)
display(candidates_grid_res)

Now process the results...

In [ ]:
name_set_groups_datasets = 'num_objects'
path_unfair_datasets = Path(f'./experiments/{name_set_groups_datasets}/')
list_files_results = sorted([f for f in path_unfair_datasets.iterdir() if f.is_file() and 'results' in f.name])
list_files_datasets = sorted([f for f in path_unfair_datasets.iterdir() if f.is_file() and 'results' not in f.name])
assert len(list_files_results) == len(list_files_datasets)


# Read the results computed over a given group of datasets from disk.
idx_tuple_list_objs = 1
dict_analyses = {}
for path_results, path_datasets in zip(list_files_results, list_files_datasets):
    print(f"Analyzing the results from {path_results} obtained from {path_datasets}")
    
    # Read the results computed over a group of datasets.
    with open(path_results, "rb") as f: set_results = pickle.load(f)
    # display(set_results)

    # Compute the number of datasets in this group.
    num_datasets_group = len(set_results['idx_candidates'])

    # Read the unfair datasets (needed to retrieve the list of object IDs belonging to the unfair hotspots).
    with open(path_datasets, "rb") as f:
        set_datasets = pickle.load(f)

    # Initialize the dictionary that will contain power, sensitivity, and PPV per subset of grids, and
    # for all the grids combined, for this group of datasets.
    # NOTE: resolution '0' means that we are considering all the subsets of grids.
    dict_analysis_res_group = {}
    for res in list_grid_res : dict_analysis_res_group[res] = \
        {'sum_power' : 0, 'sum_sensitivity' : 0., 'sum_ppv': 0.}


    # 2 - Compute power, sensitivity and PPV over this dataset.
    for idx_dataset in tqdm(range(num_datasets_group)) :

        # Retrieve the lists of object IDs associated with the various hotspots in this unfair dataset.
        # Each hotspot's list is a 1D numpy array, so we need to concatenate these arrays.
        # Guarantee also that the IDs in the final list are unique.
        set_unfair_obj_ids = np.unique(np.concatenate(set_datasets['data'][idx_dataset][idx_tuple_list_objs]))

        # Retrieve the candidates detected and thus deemed 'extreme' by the assessment approach
        # for this dataset.
        set_detected_candidates = set_results['idx_candidates'][idx_dataset]


        # 2.1 - Select the detected candidates that belong to a grid with resolution 'res'.
        for res in list_grid_res :
            
            # NOTE: if res == 0, then we consider all the candidates from all the grids.
            sel_set_detected_candidates = set_detected_candidates.copy()
            if res : sel_set_detected_candidates = \
                np.array([candidate for candidate in set_detected_candidates if candidates_grid_res[candidate] == res])

            # Retrieve the IDs of the objects associated with the extreme candidates found by the assessment
            # approach.
            list_detected_cand_objs = \
                [flattened_list_candidates[start_pos_candidates[i] : end_pos_candidates[i]] for i in sel_set_detected_candidates]
            list_detected_cand_objs = np.unique(np.concatenate(list_detected_cand_objs)) if list_detected_cand_objs else np.empty(0)

            # Compute the set intersection between the set of true unfair object IDs and the set of object IDs associated with
            # the candidates deemed 'extreme' by the assessment approach.
            intersect_obj_ids = np.intersect1d(set_unfair_obj_ids, list_detected_cand_objs)


            # Compute sensitivity and PPV. Recall:
            # - sensitivity:  measures the fraction of truly affected objects that are correctly flagged by an assessment approach
            # - ppv: measures the fraction of objects flagged by an assessment approach that truly belong to the true set.
            sensitivity = intersect_obj_ids.size / set_unfair_obj_ids.size
            ppv = intersect_obj_ids.size / list_detected_cand_objs.size if list_detected_cand_objs.size else 0.


            dict_analysis_res_group[res]['sum_power'] += (sel_set_detected_candidates.size != 0)
            dict_analysis_res_group[res]['sum_sensitivity'] += sensitivity
            dict_analysis_res_group[res]['sum_ppv'] += ppv


    # 3 - For each subset of grids of a given resolution, compute the statistical power, sensitivity, and PPV over the dataset group.
    print(f"DEBUG: Analysed group of datasets in {path_results.name}...")
    for res in list_grid_res :
        # Compute sensitivity and PPV for this resolution.
        #
        # NOTE: we skip the two divisions if dict_analysis_res_group[res]['sum_power'] == 0, because in these
        #       cases 'dict_analysis_res_group[res]['sum_sensitivity']' and 'dict_analysis_res_group[res]['sum_sensitivity']'
        #       are guaranteed to be already zero.
        if dict_analysis_res_group[res]['sum_power'] :
            dict_analysis_res_group[res]['sum_sensitivity'] = \
                dict_analysis_res_group[res]['sum_sensitivity'] / dict_analysis_res_group[res]['sum_power']
            dict_analysis_res_group[res]['sum_ppv'] = \
                dict_analysis_res_group[res]['sum_ppv'] / dict_analysis_res_group[res]['sum_power']

        # Compute the statistical power of the approach for this resolution.
        dict_analysis_res_group[res]['sum_power'] = \
            dict_analysis_res_group[res]['sum_power'] / num_datasets_group
        
        print(f"DEBUG: Power, sensitivity and PPV for grids with resolution {res}: {dict_analysis_res_group[res]['sum_power']}, " +
              f"{dict_analysis_res_group[res]['sum_sensitivity']}, " +
              f"{dict_analysis_res_group[res]['sum_ppv']}")
        

    # Add the analyses of the results from this file to the main dictionary.
    dict_analyses[path_results.name] = dict_analysis_res_group
    # break

In [ ]:
import pandas as pd
import re


# Put all the results gathered from a set of groups of dataset in a pandas dataframe.
final_df = []
idx_parameter = 0
for k,v in dict_analyses.items() :
    params = re.findall(r'\d+', k)

    dataframe_res = pd.DataFrame.from_dict(v).T
    dataframe_res['parameter'] = params[idx_parameter]
    final_df.append(dataframe_res)
final_df = pd.concat(final_df)
final_df.index.name = 'grid_res'
final_df.rename(index={0:'All'}, inplace=True)


# Now from the dataframe print the latex table to be copy/pasted in the paper.
grouped_df = final_df.groupby(final_df.index)
for key, item in grouped_df:
    # display(item)

    print("\\multirow{" + str(len(item)) + "}" + \
          "{*}{\\rotatebox[origin=c]{90}{\\textbf{" + \
          (str(key) if key else "All") + "}}}")
    
    for parameter, power, sensitivity, ppv in zip(item['parameter'], item['sum_power'], item['sum_sensitivity'], item['sum_ppv']) :
        print("& $\\mathbf{" + parameter + "}$ " +  f"& {power:.3f} & {sensitivity:.3f} & {ppv:.3f} \\\\")
    print("\\midrule")
print("\\bottomrule")
    
# display(final_df)